# P01 — Baseline Recovery & FP32 Batch-Size Audit

**Project:** RTC-SDD Research  
**Execution date:** 20 October 2026  
**Scope:** Reproduce the environment, inspect official/P01 training differences, and evaluate physical batch sizes 4, 8 and 16 using synthetic inputs and real RTCFake training audio.

> **Archived research notebook.** This is a cleaned record of a *completed diagnostic session*, **not** a new fine-tuning experiment. The original code cells have been preserved; their historical execution logs have been summarized below. No model weights or optimizer checkpoints from these smoke tests were saved.

## Reference environment

| Component | Recorded value |
|---|---|
| GitHub repository | `AhmedLabib1/RTC-SDD-Research` (private) |
| Repository revision | `d31b5abd7fe3f547423caff2a3516a42428882a6` |
| Official RTC-SDD pinned revision | `bb68fcb57672439180b791c43f4496e924efc3c1` |
| Pinned Fairseq revision | `a54021305d6b3c4c5959ac9395135f63202db8f1` |
| Notebook kernel / isolated training environment | Kaggle Python 3.13 / venv Python 3.10.21 |
| PyTorch / CUDA runtime | 2.5.1+cu121 / CUDA 12.1 |
| GPU measured | NVIDIA Tesla T4, `cuda:0` (~14.56 GiB) |
| Pretrained XLS-R checkpoint | `xlsr2_300m.pt` (original, 3,808,868,242 bytes) |
| Pretrained SHA256 | `b08927597f2c9eb2ebd7dcc3ac78ee4b5f6021cbac4b3a6c5a9deec445d80ed9` |
| Dataset | Authorized private RTCFake packed Train/Dev archives |

### Historical P01 training configuration

`Physical batch = 2`, `gradient accumulation = 8`, `effective batch = 16`, FP32, Adam, learning rate `1e-6`, weight decay `1e-4`, class weights `[0.1, 0.9]`, global gradient clipping at `1.0`, RawBoost Algorithm 5 on training only.

**Re-execution caveats:** Running this notebook from top to bottom may reinstall large dependencies and download the ~3.55 GiB original XLS-R checkpoint; later steps **consume GPU time**. Kaggle Secrets, private dataset access, suitable disk space, network access for setup/download and a GPU runtime are required. These historical, experiment-specific setup cells will be refactored into reusable framework commands in a *separate future change*. Do not treat this notebook as the reusable bootstrap implementation.


## Step 1 — Kaggle runtime and private dataset preflight

Check working storage, private RTCFake packed archives and presence (not contents) of Kaggle credentials. This step does **not** use the GPU. Never print token values into notebook outputs.


In [ ]:
from pathlib import Path
from kaggle_secrets import UserSecretsClient
import shutil
import sys

print("=" * 65)
print("STEP 1 — RTC-SDD NEW KAGGLE ENVIRONMENT")
print("=" * 65)

WORKDIR = Path("/kaggle/working")

DATASET_ROOT = Path(
    "/kaggle/input/datasets/a7medlabib/rtcfake-packed-private"
)

REPO_ROOT = WORKDIR / "RTC-SDD-Research"

print("\n[1] ENVIRONMENT")
print("Python:", sys.version.split()[0])
print("Working directory:", WORKDIR)
print(
    "Free disk:",
    round(shutil.disk_usage(WORKDIR).free / 1024**3, 2),
    "GiB"
)

print("\n[2] DATASET")

required_files = [
    "train_label.txt",
    "dev_label.txt",
    "train_online.pack",
    "train_offline.pack",
    "dev_online.pack",
    "dev_offline.pack",
]

assert DATASET_ROOT.is_dir(), "Dataset directory missing"

for filename in required_files:
    path = DATASET_ROOT / filename
    assert path.is_file(), f"Missing: {filename}"
    print("[PASS]", filename)

print("\n[3] KAGGLE SECRETS")

client = UserSecretsClient()

def check_secret(name):
    try:
        value = client.get_secret(name)
        return bool(value)
    except Exception:
        return False

github_ready = check_secret("GITHUB_TOKEN")
hf_ready = check_secret("HF_TOKEN")

print("GITHUB_TOKEN:", "READY" if github_ready else "MISSING")
print("HF_TOKEN    :", "READY" if hf_ready else "MISSING")

print("\n[4] REPOSITORY")
print("Already cloned:", (REPO_ROOT / ".git").exists())

print("\n" + "=" * 65)

if github_ready and hf_ready:
    print("[PASS] READY FOR REPOSITORY RESTORATION")
else:
    print("[ACTION REQUIRED] Enable missing Kaggle Secrets")

print("[PASS] No GPU or training used")
print("[PASS] No checkpoints modified")

## Step 2 — Restore the private repository

Clone the pinned research project from GitHub if not already present and inspect revision, branch and required files. Historical repository URL is intentionally preserved here; general Git operations belong in the reusable framework planned next.


In [ ]:
from pathlib import Path
from kaggle_secrets import UserSecretsClient
import subprocess
import tempfile
import os

ROOT = Path("/kaggle/working")
REPO = ROOT / "RTC-SDD-Research"

URL = "https://github.com/AhmedLabib1/RTC-SDD-Research.git"

print("=" * 65)
print("STEP 2 — RESTORE RTC-SDD RESEARCH REPOSITORY")
print("=" * 65)

if not REPO.exists():

    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    assert token, "GITHUB_TOKEN is missing"

    # Temporary authentication helper.
    # The token is never embedded in the Git URL.
    with tempfile.TemporaryDirectory() as temp:

        askpass = Path(temp) / "git_askpass.sh"

        askpass.write_text(
            '#!/bin/sh\n'
            'case "$1" in\n'
            '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
            '  *Password*) printf "%s\\n" "$RTC_GH_TOKEN" ;;\n'
            '  *) exit 1 ;;\n'
            'esac\n'
        )

        askpass.chmod(0o700)

        env = os.environ.copy()

        env["GIT_ASKPASS"] = str(askpass)
        env["GIT_TERMINAL_PROMPT"] = "0"
        env["RTC_GH_TOKEN"] = token

        result = subprocess.run(
            [
                "git",
                "-c", "credential.helper=",
                "clone",
                "--depth", "1",
                "--branch", "main",
                URL,
                str(REPO),
            ],
            env=env,
            capture_output=True,
            text=True,
        )

        if result.returncode != 0:
            raise RuntimeError(
                "Git clone failed. Verify token access "
                "to the private repository."
            )

    del token

    print("[PASS] Private repository cloned")

else:
    assert (REPO / ".git").is_dir(), (
        "Directory exists but is not a Git repository"
    )
    print("[INFO] Repository already exists")


def git(*args):
    return subprocess.check_output(
        ["git", "-C", str(REPO), *args],
        text=True,
    ).strip()


print("\n[1] GIT VERIFICATION")

print("Branch :", git("branch", "--show-current"))
print("Commit :", git("rev-parse", "HEAD"))
print("Status :", git("status", "--porcelain") or "CLEAN")

assert git("branch", "--show-current") == "main"

print("\n[2] REQUIRED SOURCE FILES")

required = [
    "README.md",
    "pyproject.toml",
    "src/rtcsdd/models/xlsr_aasist.py",
    "src/rtcsdd/training/xlsr_aasist_finetune.py",
    "notebooks/02_p01_xlsr_aasist_audit_and_epoch2_continuation.ipynb",
    "docs/p01_epoch1_epoch2_audit.md",
]

for filename in required:
    path = REPO / filename
    assert path.is_file(), f"Missing: {filename}"
    print("[PASS]", filename)

print("\n" + "=" * 65)
print("[PASS] RESEARCH REPOSITORY RESTORED")
print("[PASS] No GPU or training performed")
print("[PASS] No checkpoints downloaded")

## Step 3 — Official RTC-SDD vs P01 source audit

Restore and verify the pinned official source. The upstream `main_train.py` uses a default **physical batch size of 16**, and its inspected optimizer path has **no explicit gradient clipping**. The inspected model source contains 2 `BatchNorm1d` and 5 `BatchNorm2d` *declaration sites* (not necessarily the final number of instantiated modules). P01 includes global gradient clipping at 1.0.


In [ ]:

from pathlib import Path
import subprocess
import sys
import json
import re

ROOT = Path("/kaggle/working/RTC-SDD-Research")

OFFICIAL = ROOT / "third_party/official_rtc_sdd"

CONFIG = (
    ROOT / "src/rtcsdd/configs/"
    "p01_xlsr_aasist_official_like_fp32.json"
)

BOOTSTRAP = ROOT / "scripts/setup/bootstrap_official.py"

print("=" * 75)
print("STEP 3 — OFFICIAL BASELINE VS P01 TRAINING AUDIT")
print("=" * 75)

# --------------------------------------------------
# 1. Restore pinned official source
# --------------------------------------------------

print("\n[1] OFFICIAL SOURCE RESTORATION")

assert BOOTSTRAP.is_file()
assert CONFIG.is_file()

subprocess.run(
    [sys.executable, str(BOOTSTRAP)],
    cwd=ROOT,
    check=True,
)

subprocess.run(
    [
        sys.executable,
        str(BOOTSTRAP),
        "--verify-only",
    ],
    cwd=ROOT,
    check=True,
)

official_commit = subprocess.check_output(
    ["git", "-C", str(OFFICIAL), "rev-parse", "HEAD"],
    text=True,
).strip()

print("[PASS] Official source restored")
print("Official commit:", official_commit)

# --------------------------------------------------
# 2. P01 configuration
# --------------------------------------------------

print("\n[2] P01 TRAINING CONFIGURATION")
print("-" * 75)

config = json.loads(CONFIG.read_text())
train = config["training"]
data = config["data"]

keys = [
    "epochs",
    "batch_size",
    "gradient_accumulation",
    "learning_rate",
    "weight_decay",
    "optimizer",
    "class_weights",
    "precision",
    "gradient_clip_norm",
    "lr_scheduler",
    "seed",
]

for key in keys:
    print(f"{key:<28}: {train.get(key)}")

print(
    "effective_batch_size".ljust(28) + ":",
    train["batch_size"] * train["gradient_accumulation"]
)

print("audio_length".ljust(28) + ":", data["audio_length"])
print("crop".ljust(28) + ":", data["crop"])
print("pad".ljust(28) + ":", data["pad"])

# --------------------------------------------------
# 3. Official training source
# --------------------------------------------------

print("\n[3] OFFICIAL TRAINING SOURCE")
print("-" * 75)

OFFICIAL_TRAIN = (
    OFFICIAL / "xlsr_aasist/main_train.py"
)

assert OFFICIAL_TRAIN.is_file()

lines = OFFICIAL_TRAIN.read_text().splitlines()

patterns = [
    "batch_size",
    "num_epochs",
    "earlystop_epoch",
    "weight_decay",
    "CrossEntropyLoss",
    "torch.optim.Adam",
    "optimizer.zero_grad",
    "batch_loss.backward",
    "optimizer.step",
    "clip_grad_norm_",
]

for number, line in enumerate(lines, start=1):

    if any(pattern in line for pattern in patterns):

        print(f"{number:04d}: {line.strip()}")

# --------------------------------------------------
# 4. BatchNorm architecture audit
# --------------------------------------------------

print("\n[4] OFFICIAL BATCHNORM ARCHITECTURE")
print("-" * 75)

MODEL_SOURCE = (
    OFFICIAL / "xlsr_aasist/model/model.py"
)

source = MODEL_SOURCE.read_text()

for dimension in [1, 2, 3]:

    pattern = rf"\bnn\.BatchNorm{dimension}d\s*\("

    count = len(re.findall(pattern, source))

    print(f"BatchNorm{dimension}d call sites: {count}")

print(
    "[INFO] Call-site counts are not necessarily "
    "the number of instantiated layers."
)

# --------------------------------------------------
# 5. P01 gradient operations
# --------------------------------------------------

print("\n[5] P01 GRADIENT HANDLING")
print("-" * 75)

files = [
    ROOT / "src/rtcsdd/training/trainer.py",
    ROOT / "src/rtcsdd/training/weighted_loss.py",
]

for path in files:

    assert path.is_file()
    print("\nFile:", path.name)

    for number, line in enumerate(
        path.read_text().splitlines(),
        start=1,
    ):

        if any(term in line for term in [
            "weighted_ce_sum(",
            "normalize_gradients(",
            "clip_grad_norm_(",
            "optimizer.step()",
            "numerator.backward()",
            "scaler.step(",
        ]):

            print(f"{number:04d}: {line.strip()}")

# --------------------------------------------------
# 6. Final status
# --------------------------------------------------

print("\n" + "=" * 75)
print("[PASS] SOURCE AUDIT COMPLETE")
print("=" * 75)

print("[PASS] Official source commit verified")
print("[PASS] P01 configuration inspected")
print("[PASS] BatchNorm declarations inspected")
print("[PASS] Gradient operations inspected")
print("[PASS] No GPU or training performed")
print("[PASS] No checkpoints downloaded")
print("[PASS] No model files modified")


## Step 4D — Restore pinned Python environment

Recreate the Python 3.10 venv if absent; apply a **temporary** compatibility conversion to the legacy Fairseq requirement without modifying the official requirements file. Install the framework and verify imports/versions. The suffix `4D` is retained from the historical debug sequence; superseded attempts are not included.


In [ ]:

from pathlib import Path
import subprocess
import hashlib
import shutil

ROOT = Path("/kaggle/working/RTC-SDD-Research")

ENV = Path("/kaggle/working/rtcsdd-baseline-py310")
PY310 = ENV / "bin/python"

OFFICIAL_REQ = (
    ROOT / "third_party/official_rtc_sdd/requirements.txt"
)

TEMP_REQ = Path(
    "/kaggle/working/rtcsdd_requirements_uv_compatible.txt"
)

FAIRSEQ_SHA = "a54021305d6b3c4c5959ac9395135f63202db8f1"

print("=" * 70)
print("STEP 4D — RESTORE & VERIFY PINNED ENVIRONMENT")
print("=" * 70)

# ==================================================
# 1. CHECK REPOSITORY
# ==================================================

print("\n[1] REPOSITORY CHECK")

assert ROOT.is_dir(), (
    "Repository missing. Run Step 2 first."
)

assert OFFICIAL_REQ.is_file(), (
    "Official source missing. Run Step 3 first."
)

print("[PASS] Repository available")
print("[PASS] Official requirements available")

# ==================================================
# 2. RESTORE PYTHON 3.10 IF MISSING
# ==================================================

print("\n[2] PYTHON ENVIRONMENT")

uv = shutil.which("uv")
assert uv is not None, "uv is not installed"

if not PY310.is_file():

    print("[INFO] Python 3.10 venv missing")
    print("[INFO] Restoring environment...", flush=True)

    subprocess.run(
        [uv, "python", "install", "3.10"],
        check=True,
    )

    subprocess.run(
        [
            uv,
            "venv",
            "--python", "3.10",
            str(ENV),
        ],
        check=True,
    )

assert PY310.is_file()

version = subprocess.check_output(
    [str(PY310), "--version"],
    text=True,
).strip()

assert version.startswith("Python 3.10.")

print("[PASS]", version)
print("[PASS] Virtual environment ready")

# ==================================================
# 3. NORMALIZE FAIRSEQ REQUIREMENT
# ==================================================

print("\n[3] PREPARING REQUIREMENTS")

original_bytes = OFFICIAL_REQ.read_bytes()
original_hash = hashlib.sha256(original_bytes).hexdigest()

lines = original_bytes.decode("utf-8").splitlines()

matches = [
    i for i, line in enumerate(lines)
    if "github.com/pytorch/fairseq" in line
    and not line.strip().startswith("#")
]

assert len(matches) == 1

idx = matches[0]

assert FAIRSEQ_SHA in lines[idx]

lines[idx] = (
    "fairseq @ git+https://github.com/pytorch/"
    f"fairseq.git@{FAIRSEQ_SHA}"
)

TEMP_REQ.write_text(
    "\n".join(lines) + "\n",
    encoding="utf-8",
)

print("[PASS] Fairseq requirement normalized")
print("[PASS] Original requirements unchanged")

# ==================================================
# 4. INSTALL DEPENDENCIES
# ==================================================

index_options = [
    "--default-index",
    "https://pypi.org/simple",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cu121",
    "--index-strategy",
    "unsafe-first-match",
]

print("\n[4] INSTALLING PINNED DEPENDENCIES", flush=True)

subprocess.run(
    [
        uv, "pip", "install",
        "--python", str(PY310),
        "-r", str(TEMP_REQ),
        *index_options,
    ],
    cwd=ROOT,
    check=True,
)

print("[PASS] Dependencies installed")

# ==================================================
# 5. INSTALL RESEARCH FRAMEWORK
# ==================================================

print("\n[5] INSTALLING RESEARCH FRAMEWORK", flush=True)

subprocess.run(
    [
        uv, "pip", "install",
        "--python", str(PY310),
        "-e", str(ROOT),
        "huggingface_hub>=0.30,<1.0",
        "soundfile>=0.12,<1",
        *index_options,
    ],
    cwd=ROOT,
    check=True,
)

print("[PASS] Research framework installed")

# ==================================================
# 6. DEPENDENCY CHECK
# ==================================================

print("\n[6] DEPENDENCY CHECK", flush=True)

subprocess.run(
    [
        uv, "pip", "check",
        "--python", str(PY310),
    ],
    check=True,
)

# ==================================================
# 7. RUNTIME VERIFICATION
# ==================================================

print("\n[7] RUNTIME VERIFICATION", flush=True)

verification = r"""
import sys
import numpy as np

for name, value in {
    "float": float,
    "int": int,
    "complex": complex,
    "bool": bool,
    "object": object,
    "str": str,
}.items():
    if name not in np.__dict__:
        setattr(np, name, value)

import torch
import fairseq
import soundfile
import rtcsdd

from rtcsdd.models.xlsr_aasist import load_xlsr_aasist

assert sys.version_info[:2] == (3, 10)
assert torch.__version__ == "2.5.1+cu121"

print("Python :", sys.version.split()[0])
print("NumPy  :", np.__version__)
print("Torch  :", torch.__version__)
print("CUDA   :", torch.version.cuda)

print("[PASS] Fairseq import")
print("[PASS] RTC-SDD import")
print("[PASS] XLS-R+AASIST loader")

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
"""

subprocess.run(
    [
        str(PY310),
        "-u",
        "-c",
        verification,
    ],
    cwd=ROOT,
    check=True,
)

# ==================================================
# 8. SOURCE INTEGRITY
# ==================================================

assert hashlib.sha256(
    OFFICIAL_REQ.read_bytes()
).hexdigest() == original_hash

print("\n" + "=" * 70)
print("[PASS] ENVIRONMENT READY")
print("[PASS] OFFICIAL REQUIREMENTS UNCHANGED")
print("[PASS] NO MODEL TRAINING PERFORMED")
print("[PASS] NO CHECKPOINTS MODIFIED")


## Step 5 — Fetch and verify original XLS-R 300M weights

Verify SHA256 and exact file size before accepting the original checkpoint. Download only when missing. This is the **original pretrained model**, not the P01 fine-tuned model; no trained weights are modified.


In [ ]:

from pathlib import Path
import hashlib
import shutil
import urllib.request

PRETRAINED = Path(
    "/kaggle/working/checkpoints/pretrained/xlsr2_300m.pt"
)

TEMP = Path(str(PRETRAINED) + ".part")

URL = (
    "https://dl.fbaipublicfiles.com/"
    "fairseq/wav2vec/xlsr2_300m.pt"
)

# Verified SHA256 for original XLS-R 300M.
EXPECTED_SHA256 = (
    "b08927597f2c9eb2ebd7dcc3ac78ee4b5f6021cbac"
    "4b3a6c5a9deec445d80ed9"
)

EXPECTED_SIZE = 3_808_868_242

print("=" * 70)
print("STEP 5 — RESTORE & VERIFY PRETRAINED XLS-R 300M")
print("=" * 70)

assert len(EXPECTED_SHA256) == 64

PRETRAINED.parent.mkdir(
    parents=True,
    exist_ok=True,
)


def verify_checkpoint(path):
    """Verify exact size and SHA256 without loading the model."""

    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(8 * 1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    actual_hash = digest.hexdigest()
    actual_size = path.stat().st_size

    print("File   :", path.name)
    print("Size   :", f"{actual_size / 1024**3:.3f} GiB")
    print("SHA256 :", actual_hash)

    assert actual_size == EXPECTED_SIZE, (
        f"File size mismatch: {actual_size}"
    )

    assert actual_hash == EXPECTED_SHA256, (
        "SHA256 mismatch. File preserved for investigation."
    )

    print("[PASS] File size and SHA256 verified")


# ==================================================
# 1. RESTORE EXISTING FILE
# ==================================================

if PRETRAINED.is_file():

    print("\n[1] EXISTING CHECKPOINT")
    verify_checkpoint(PRETRAINED)

elif TEMP.is_file():

    print("\n[1] RECOVER PREVIOUS DOWNLOAD")
    print("[INFO] Temporary download found")
    print("[INFO] No new download required")

    verify_checkpoint(TEMP)

    TEMP.replace(PRETRAINED)

    print("[PASS] Temporary file promoted to final checkpoint")

else:

    # ==================================================
    # 2. DOWNLOAD ONLY IF NO FILE EXISTS
    # ==================================================

    print("\n[1] DOWNLOAD REQUIRED")

    free_gib = (
        shutil.disk_usage("/kaggle/working").free
        / 1024**3
    )

    print(f"Free disk: {free_gib:.2f} GiB")

    assert free_gib >= 5.0, (
        "Insufficient free disk space"
    )

    print("\nDownloading XLS-R 300M...", flush=True)

    request = urllib.request.Request(
        URL,
        headers={"User-Agent": "RTC-SDD-Research/1.0"},
    )

    downloaded = 0
    last_report = 0

    with urllib.request.urlopen(
        request,
        timeout=180,
    ) as response:

        with TEMP.open("wb") as output:

            while True:

                chunk = response.read(
                    8 * 1024 * 1024
                )

                if not chunk:
                    break

                output.write(chunk)
                downloaded += len(chunk)

                if downloaded - last_report >= 256 * 1024 * 1024:

                    print(
                        f"[DOWNLOAD] "
                        f"{downloaded / 1024**3:.2f} GiB",
                        flush=True,
                    )

                    last_report = downloaded

    print("\n[2] VERIFY DOWNLOADED FILE")
    verify_checkpoint(TEMP)

    TEMP.replace(PRETRAINED)

    print("[PASS] Download completed and verified")


# ==================================================
# 3. FINAL RESULT
# ==================================================

print("\n" + "=" * 70)
print("FINAL CHECKPOINT VERIFICATION")
print("=" * 70)

assert PRETRAINED.is_file()
assert PRETRAINED.stat().st_size == EXPECTED_SIZE

print("Path:", PRETRAINED)
print("Expected SHA256:", EXPECTED_SHA256)

print("\n[PASS] PRETRAINED XLS-R 300M READY")
print("[PASS] FILE INTEGRITY VERIFIED")
print("[PASS] NO GPU OR TRAINING PERFORMED")
print("[PASS] ORIGINAL P01 CHECKPOINTS UNTOUCHED")


## Step 6 — Synthetic FP32 memory check (batch 4)

Use random waveforms and simulated Adam moment buffers to estimate training-mode GPU memory. This step performs forward/backward but **no optimizer update**. Its results are memory diagnostics, not model performance or full training validation.


In [ ]:

from pathlib import Path
import subprocess

ROOT = Path("/kaggle/working/RTC-SDD-Research")

PY310 = Path(
    "/kaggle/working/rtcsdd-baseline-py310/bin/python"
)

OFFICIAL = ROOT / "third_party/official_rtc_sdd"

PRETRAINED = Path(
    "/kaggle/working/checkpoints/pretrained/xlsr2_300m.pt"
)

print("=" * 70)
print("STEP 6 — FP32 PHYSICAL BATCH SIZE MEMORY TEST")
print("=" * 70)

# --------------------------------------------------
# 1. Runtime preflight
# --------------------------------------------------

required = [
    PY310,
    PRETRAINED,
    OFFICIAL / "xlsr_aasist/model/model.py",
]

for path in required:
    assert path.is_file(), (
        f"Missing: {path}\n"
        "The Kaggle runtime may have restarted. "
        "Restore missing files before continuing."
    )

print("[PASS] Runtime files available")

# --------------------------------------------------
# 2. Isolated GPU memory test
# --------------------------------------------------

script = r"""
import sys
import gc
import torch
import torch.nn.functional as F
from pathlib import Path

from rtcsdd.models.xlsr_aasist import load_xlsr_aasist

OFFICIAL = Path(sys.argv[1])
PRETRAINED = Path(sys.argv[2])

DEVICE = "cuda:0"
BATCH_SIZE = 4
AUDIO_LENGTH = 64600
GIB = 1024 ** 3

print("\n[1] GPU CHECK", flush=True)

assert torch.cuda.is_available(), (
    "CUDA unavailable. Enable GPU T4 x2."
)

torch.cuda.set_device(0)

print("GPU:", torch.cuda.get_device_name(0))
print(
    "Total VRAM:",
    round(torch.cuda.get_device_properties(0).total_memory / GIB, 2),
    "GiB"
)

# --------------------------------------------------
# Load actual pretrained architecture
# --------------------------------------------------

print("\n[2] LOADING MODEL", flush=True)

model = load_xlsr_aasist(
    OFFICIAL,
    PRETRAINED,
    DEVICE,
)

for param in model.parameters():
    param.requires_grad_(True)

model.train()

assert next(model.parameters()).is_cuda

parameters = [
    p for p in model.parameters()
    if p.requires_grad
]

total_params = sum(p.numel() for p in parameters)

print("Trainable parameters:", f"{total_params:,}")
print("[PASS] Model in training mode")

# --------------------------------------------------
# Reserve approximate Adam optimizer state memory.
# No optimizer updates are performed.
# --------------------------------------------------

print("\n[3] SIMULATING ADAM STATE MEMORY", flush=True)

adam_buffers = []

for param in parameters:
    adam_buffers.append((
        torch.zeros_like(param),
        torch.zeros_like(param),
    ))

torch.cuda.synchronize()

print(
    "Adam state storage:",
    round(sum(
        a.numel() * a.element_size()
        + b.numel() * b.element_size()
        for a, b in adam_buffers
    ) / GIB, 2),
    "GiB"
)

# --------------------------------------------------
# FP32 forward and backward
# --------------------------------------------------

print("\n[4] FORWARD + BACKWARD", flush=True)

torch.manual_seed(1234)

audio = (
    torch.randn(
        BATCH_SIZE,
        AUDIO_LENGTH,
        dtype=torch.float32,
        device=DEVICE,
    ) * 0.03
)

labels = torch.tensor(
    [0, 1, 0, 1],
    dtype=torch.long,
    device=DEVICE,
)

weights = torch.tensor(
    [0.1, 0.9],
    dtype=torch.float32,
    device=DEVICE,
)

torch.cuda.reset_peak_memory_stats()

logits = model(audio)

assert logits.shape == (BATCH_SIZE, 2)

loss = F.cross_entropy(
    logits.float(),
    labels,
    weight=weights,
)

assert torch.isfinite(loss), "Non-finite loss"

loss.backward()

torch.cuda.synchronize()

# --------------------------------------------------
# Memory measurements
# --------------------------------------------------

peak_allocated = (
    torch.cuda.max_memory_allocated() / GIB
)

peak_reserved = (
    torch.cuda.max_memory_reserved() / GIB
)

free_bytes, total_bytes = torch.cuda.mem_get_info()

free_gib = free_bytes / GIB

print("\n" + "=" * 65)
print("GPU MEMORY TEST RESULTS")
print("=" * 65)

print("Physical batch size :", BATCH_SIZE)
print("Precision           : FP32")
print("Training mode       : True")
print("Forward             : PASS")
print("Backward            : PASS")
print("Loss finite         : PASS")

print(
    "Peak allocated VRAM:",
    f"{peak_allocated:.2f} GiB"
)
print(
    "Peak reserved VRAM :",
    f"{peak_reserved:.2f} GiB"
)
print(
    "Free VRAM after test:",
    f"{free_gib:.2f} GiB"
)

print("\n[PASS] BATCH 4 FORWARD/BACKWARD FITS")

if free_gib < 2.0:
    print(
        "[WARNING] Limited remaining VRAM. "
        "Full Adam update may require more memory."
    )
else:
    print(
        "[INFO] Preliminary memory headroom available"
    )

# No optimizer.step() is ever called.
model.zero_grad(set_to_none=True)

del audio, labels, logits, loss
del adam_buffers, model

gc.collect()
torch.cuda.empty_cache()

print("\n[PASS] No optimizer updates performed")
print("[PASS] No checkpoints modified")
print("[INFO] Full training feasibility not yet proven")
"""

subprocess.run(
    [
        str(PY310),
        "-u",
        "-c",
        script,
        str(OFFICIAL),
        str(PRETRAINED),
    ],
    cwd=ROOT,
    check=True,
)

print("\n[PASS] STEP 6 COMPLETE")


## Step 7 — Synthetic memory checks (batches 8 and 16)

Repeat synthetic forward/backward in independent subprocesses, including simulated Adam moment storage. The tests do not run a real Adam step or demonstrate multi-epoch stability.


In [ ]:

from pathlib import Path
import subprocess

ROOT = Path("/kaggle/working/RTC-SDD-Research")

PY310 = Path(
    "/kaggle/working/rtcsdd-baseline-py310/bin/python"
)

OFFICIAL = ROOT / "third_party/official_rtc_sdd"

PRETRAINED = Path(
    "/kaggle/working/checkpoints/pretrained/xlsr2_300m.pt"
)

print("=" * 70)
print("STEP 7 — PHYSICAL BATCH 8 / 16 MEMORY FEASIBILITY")
print("=" * 70)

for path in [
    PY310,
    PRETRAINED,
    OFFICIAL / "xlsr_aasist/model/model.py",
]:
    assert path.is_file(), f"Missing: {path}"

script = r"""
import sys
import torch
import torch.nn.functional as F

from pathlib import Path
from rtcsdd.models.xlsr_aasist import load_xlsr_aasist

OFFICIAL = Path(sys.argv[1])
PRETRAINED = Path(sys.argv[2])
BATCH = int(sys.argv[3])

DEVICE = "cuda:0"
GIB = 1024 ** 3

try:
    assert torch.cuda.is_available()

    torch.cuda.set_device(0)
    torch.manual_seed(1234)
    torch.backends.cudnn.benchmark = False

    print("GPU:", torch.cuda.get_device_name(0))
    print("Physical batch:", BATCH)

    model = load_xlsr_aasist(
        OFFICIAL,
        PRETRAINED,
        DEVICE,
    )

    model.train()

    parameters = list(model.parameters())

    for p in parameters:
        p.requires_grad_(True)

    # Simulate FP32 Adam first/second moment storage.
    # The test does not initialize or update an optimizer.
    adam_state = [
        (torch.zeros_like(p), torch.zeros_like(p))
        for p in parameters
    ]

    torch.cuda.reset_peak_memory_stats()

    audio = torch.randn(
        BATCH,
        64600,
        device=DEVICE,
        dtype=torch.float32,
    ) * 0.03

    labels = (
        torch.arange(BATCH, device=DEVICE) % 2
    ).long()

    weights = torch.tensor(
        [0.1, 0.9],
        device=DEVICE,
        dtype=torch.float32,
    )

    print("[RUNNING] Forward...", flush=True)

    logits = model(audio)

    assert logits.shape == (BATCH, 2)

    loss = F.cross_entropy(
        logits.float(),
        labels,
        weight=weights,
    )

    assert torch.isfinite(loss).item()

    print("[RUNNING] Backward...", flush=True)

    loss.backward()

    torch.cuda.synchronize()

    allocated = (
        torch.cuda.max_memory_allocated() / GIB
    )

    reserved = (
        torch.cuda.max_memory_reserved() / GIB
    )

    free, total = torch.cuda.mem_get_info()

    print("\nRESULTS")
    print("Batch:", BATCH)
    print("Forward: PASS")
    print("Backward: PASS")
    print("Peak allocated:", f"{allocated:.2f} GiB")
    print("Peak reserved :", f"{reserved:.2f} GiB")
    print("Free VRAM     :", f"{free / GIB:.2f} GiB")

    print(f"[PASS] BATCH {BATCH} MEMORY TEST")
    print("[INFO] No optimizer update performed")

except torch.cuda.OutOfMemoryError as exc:
    print(
        f"[OOM] Batch {BATCH} exceeds available "
        "VRAM during this test",
        flush=True,
    )
    sys.exit(42)
"""

results = {}

for batch in [8, 16]:

    print("\n" + "=" * 70)
    print(f"TESTING PHYSICAL BATCH SIZE = {batch}")
    print("=" * 70, flush=True)

    result = subprocess.run(
        [
            str(PY310),
            "-u",
            "-c",
            script,
            str(OFFICIAL),
            str(PRETRAINED),
            str(batch),
        ],
        cwd=ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    # Keep output readable.
    print("\n".join(result.stdout.splitlines()[-30:]))

    if result.returncode == 0:
        results[batch] = "PASS"

    elif result.returncode == 42:
        results[batch] = "OOM"
        print("[INFO] Skipping larger batch sizes")
        break

    else:
        raise RuntimeError(
            f"Unexpected failure at batch {batch}. "
            "Inspect output before continuing."
        )

print("\n" + "=" * 70)
print("FINAL MEMORY TEST SUMMARY")
print("=" * 70)

print("Batch 2  : Previously trained in P01")
print("Batch 4  : PASS (Step 6)")

for batch, status in results.items():
    print(f"Batch {batch:<2} : {status}")

print("\n[PASS] Memory feasibility tests finished")
print("[PASS] No optimizer updates performed")
print("[PASS] No checkpoints modified")
print("[INFO] Full training feasibility still unverified")


## Step 8 — Real-audio optimizer smoke test (batch 8 × accumulation 2)

Use **16 actual RTCFake Train recordings**, RawBoost Algorithm 5, FP32 and one real Adam update with effective batch 16. All changes remain in process memory. No P01 checkpoints are written or resumed.


In [ ]:

from pathlib import Path
import subprocess

ROOT = Path("/kaggle/working/RTC-SDD-Research")
PY310 = Path("/kaggle/working/rtcsdd-baseline-py310/bin/python")

DATA = Path(
    "/kaggle/input/datasets/a7medlabib/rtcfake-packed-private"
)

PRETRAINED = Path(
    "/kaggle/working/checkpoints/pretrained/xlsr2_300m.pt"
)

CONFIG = ROOT / (
    "src/rtcsdd/configs/"
    "p01_xlsr_aasist_official_like_fp32.json"
)

for path in [PY310, PRETRAINED, CONFIG, DATA / "train_label.txt"]:
    assert path.is_file(), f"Missing: {path}"

script = r"""
import sys
import json
import time
import math

from pathlib import Path

import torch
from torch.utils.data import DataLoader, Subset

from rtcsdd.data.protocol import read_protocol
from rtcsdd.data.dataset import RTCFakePackedDataset
from rtcsdd.models.xlsr_aasist import load_xlsr_aasist
from rtcsdd.training.trainer import epoch_permutation
from rtcsdd.training.weighted_loss import (
    weighted_ce_sum,
    normalize_gradients,
)

ROOT, DATA, PRETRAINED, CONFIG = map(Path, sys.argv[1:])

cfg = json.loads(CONFIG.read_text())
train = cfg["training"]
data = cfg["data"]

BATCH = 8
ACCUM = 2
DEVICE = "cuda:0"
GIB = 1024 ** 3

print("=" * 65, flush=True)
print("STEP 8 — REAL RTCFAKE TRAINING SMOKE TEST")
print("=" * 65, flush=True)

assert torch.cuda.is_available()
assert train["precision"] == "fp32"
assert BATCH * ACCUM == 16

torch.cuda.set_device(0)
torch.manual_seed(train["seed"])

torch.backends.cudnn.benchmark = False

# --------------------------------------------------
# 1. Actual RTCFake Train recordings
# --------------------------------------------------

print("\n[1] LOADING REAL TRAIN DATA", flush=True)

examples = read_protocol(DATA / "train_label.txt", "train")

assert len(examples) == 75785

dataset = RTCFakePackedDataset(
    examples,
    DATA,
    length=data["audio_length"],
    crop=data["crop"],
    pad=data["pad"],
    rawboost=data["rawboost"],
    upstream_root=ROOT / "third_party/official_rtc_sdd",
    seed=train["seed"],
)

dataset.set_epoch(0)

indices = epoch_permutation(
    len(dataset), train["seed"], 0
)[:BATCH * ACCUM]

loader = DataLoader(
    Subset(dataset, indices),
    batch_size=BATCH,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

print("Real audio samples:", len(indices))
print("RawBoost enabled:", data["rawboost"]["enabled"])
print("Physical batch:", BATCH)
print("Accumulation:", ACCUM)

# --------------------------------------------------
# 2. Load actual model
# --------------------------------------------------

print("\n[2] LOADING MODEL", flush=True)

model = load_xlsr_aasist(
    ROOT / "third_party/official_rtc_sdd",
    PRETRAINED,
    DEVICE,
)

model.train()

parameters = list(model.parameters())

for p in parameters:
    p.requires_grad_(True)

optimizer = torch.optim.Adam(
    parameters,
    lr=train["learning_rate"],
    weight_decay=train["weight_decay"],
)

weights = torch.tensor(
    train["class_weights"],
    device=DEVICE,
    dtype=torch.float32,
)

optimizer.zero_grad(set_to_none=True)

# --------------------------------------------------
# 3. Real training update
# --------------------------------------------------

print("\n[3] REAL FORWARD / BACKWARD", flush=True)

torch.cuda.reset_peak_memory_stats()
started = time.monotonic()

weight_sum = 0.0

for step, (audio, labels, ids) in enumerate(loader, 1):

    assert audio.shape == (BATCH, 64600)
    assert torch.isfinite(audio).all()

    audio = audio.to(DEVICE, non_blocking=True)
    labels = labels.to(DEVICE, non_blocking=True)

    logits = model(audio)

    numerator, denominator = weighted_ce_sum(
        logits,
        labels,
        weights,
    )

    assert torch.isfinite(numerator).item()

    numerator.backward()

    weight_sum += denominator.item()

    torch.cuda.synchronize()

    print(
        f"[PASS] Micro-batch {step}/{ACCUM} "
        f"| Size={BATCH} "
        f"| Peak VRAM="
        f"{torch.cuda.max_memory_reserved()/GIB:.2f} GiB",
        flush=True,
    )

assert step == ACCUM

# --------------------------------------------------
# 4. Gradient normalization and actual Adam step
# --------------------------------------------------

print("\n[4] ADAM OPTIMIZER UPDATE", flush=True)

grad_norm = normalize_gradients(
    parameters,
    weight_sum=weight_sum,
    max_norm=train["gradient_clip_norm"],
)

assert math.isfinite(grad_norm), (
    "Non-finite gradient norm; optimizer step blocked"
)

optimizer.step()

torch.cuda.synchronize()

elapsed = time.monotonic() - started

peak_allocated = torch.cuda.max_memory_allocated() / GIB
peak_reserved = torch.cuda.max_memory_reserved() / GIB

free, total = torch.cuda.mem_get_info()

print("\n" + "=" * 65)
print("RESULTS")
print("=" * 65)

print("Physical Batch :", BATCH)
print("Accumulation   :", ACCUM)
print("Effective Batch:", BATCH * ACCUM)
print("Precision      : FP32")
print("RawBoost       : ON")
print("Real Audio     : YES")
print("Optimizer Steps: 1")
print("Gradient Norm  :", f"{grad_norm:.4f}")
print("Runtime        :", f"{elapsed:.2f} seconds")
print("Peak Allocated :", f"{peak_allocated:.2f} GiB")
print("Peak Reserved  :", f"{peak_reserved:.2f} GiB")
print("Free VRAM      :", f"{free/GIB:.2f} GiB")

print("\n[PASS] REAL TRAINING SMOKE TEST COMPLETED")
print("[PASS] ADAM UPDATE EXECUTED")
print("[PASS] No checkpoints saved")
print("[PASS] No changes to existing P01 experiment")
"""

completed = subprocess.run(
    [
        str(PY310),
        "-u",
        "-c",
        script,
        str(ROOT),
        str(DATA),
        str(PRETRAINED),
        str(CONFIG),
    ],
    cwd=ROOT,
    check=True,
)


## Step 9 — Real-audio optimizer smoke test (batch 16 × accumulation 1)

Use the same total of 16 authorized real Train recordings in a single physical batch, one Adam update and RawBoost 5. This is still a one-update diagnostic, **not a full-epoch training result**.


In [ ]:

from pathlib import Path
import subprocess

ROOT = Path("/kaggle/working/RTC-SDD-Research")

PY310 = Path(
    "/kaggle/working/rtcsdd-baseline-py310/bin/python"
)

DATA = Path(
    "/kaggle/input/datasets/a7medlabib/rtcfake-packed-private"
)

PRETRAINED = Path(
    "/kaggle/working/checkpoints/pretrained/xlsr2_300m.pt"
)

CONFIG = (
    ROOT / "src/rtcsdd/configs/"
    "p01_xlsr_aasist_official_like_fp32.json"
)

for path in [PY310, PRETRAINED, CONFIG, DATA / "train_label.txt"]:
    assert path.is_file(), f"Missing: {path}"

script = r"""
import sys
import json
import time
import math

from pathlib import Path

import torch
from torch.utils.data import DataLoader, Subset

from rtcsdd.data.protocol import read_protocol
from rtcsdd.data.dataset import RTCFakePackedDataset
from rtcsdd.models.xlsr_aasist import load_xlsr_aasist
from rtcsdd.training.trainer import epoch_permutation
from rtcsdd.training.weighted_loss import (
    weighted_ce_sum,
    normalize_gradients,
)

ROOT, DATA, PRETRAINED, CONFIG = map(Path, sys.argv[1:])

cfg = json.loads(CONFIG.read_text())
train = cfg["training"]
data = cfg["data"]

BATCH = 16
ACCUM = 1
DEVICE = "cuda:0"
GIB = 1024 ** 3

print("=" * 65, flush=True)
print("STEP 9 — REAL BATCH 16 TRAINING SMOKE TEST")
print("=" * 65, flush=True)

try:
    assert torch.cuda.is_available()
    assert train["precision"] == "fp32"

    torch.cuda.set_device(0)
    torch.manual_seed(train["seed"])
    torch.cuda.manual_seed_all(train["seed"])

    torch.backends.cudnn.benchmark = False

    print("\n[1] REAL RTCFAKE DATA", flush=True)

    examples = read_protocol(
        DATA / "train_label.txt", "train"
    )

    indices = epoch_permutation(
        len(examples), train["seed"], 0
    )[:BATCH]

    dataset = RTCFakePackedDataset(
        examples,
        DATA,
        length=data["audio_length"],
        crop=data["crop"],
        pad=data["pad"],
        rawboost=data["rawboost"],
        upstream_root=ROOT / "third_party/official_rtc_sdd",
        seed=train["seed"],
    )

    dataset.set_epoch(0)

    loader = DataLoader(
        Subset(dataset, indices),
        batch_size=BATCH,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )

    print("Real samples:", len(indices))
    print("Physical batch:", BATCH)
    print("Accumulation:", ACCUM)
    print("RawBoost: ON")

    print("\n[2] MODEL LOADING", flush=True)

    model = load_xlsr_aasist(
        ROOT / "third_party/official_rtc_sdd",
        PRETRAINED,
        DEVICE,
    )

    model.train()

    parameters = list(model.parameters())

    for p in parameters:
        p.requires_grad_(True)

    optimizer = torch.optim.Adam(
        parameters,
        lr=train["learning_rate"],
        weight_decay=train["weight_decay"],
    )

    weights = torch.tensor(
        train["class_weights"],
        device=DEVICE,
        dtype=torch.float32,
    )

    optimizer.zero_grad(set_to_none=True)

    torch.cuda.reset_peak_memory_stats()
    started = time.monotonic()

    weight_sum = 0.0

    print("\n[3] FORWARD + BACKWARD", flush=True)

    for audio, labels, ids in loader:

        assert audio.shape == (
            BATCH, data["audio_length"]
        )

        audio = audio.to(
            DEVICE, non_blocking=True
        )

        labels = labels.to(
            DEVICE, non_blocking=True
        )

        logits = model(audio)

        numerator, denominator = weighted_ce_sum(
            logits,
            labels,
            weights,
        )

        assert torch.isfinite(numerator).item()

        numerator.backward()
        weight_sum += denominator.item()

    print("[PASS] Forward / Backward", flush=True)

    print("\n[4] ADAM UPDATE", flush=True)

    grad_norm = normalize_gradients(
        parameters,
        weight_sum=weight_sum,
        max_norm=train["gradient_clip_norm"],
    )

    assert math.isfinite(grad_norm)

    optimizer.step()
    torch.cuda.synchronize()

    elapsed = time.monotonic() - started

    allocated = (
        torch.cuda.max_memory_allocated() / GIB
    )

    reserved = (
        torch.cuda.max_memory_reserved() / GIB
    )

    free, total = torch.cuda.mem_get_info()
    free_gib = free / GIB

    print("\n" + "=" * 65)
    print("RESULTS")
    print("=" * 65)

    print("Physical Batch :", BATCH)
    print("Accumulation   :", ACCUM)
    print("Effective Batch:", BATCH * ACCUM)
    print("Precision      : FP32")
    print("Real Audio     : YES")
    print("RawBoost       : ON")
    print("Optimizer Steps: 1")
    print("Gradient Norm  :", f"{grad_norm:.4f}")
    print("Runtime        :", f"{elapsed:.2f} seconds")
    print("Peak Allocated :", f"{allocated:.2f} GiB")
    print("Peak Reserved  :", f"{reserved:.2f} GiB")
    print("Free VRAM      :", f"{free_gib:.2f} GiB")

    print("\n[PASS] BATCH 16 ADAM UPDATE COMPLETED")

    if free_gib < 2.0:
        print("[WARNING] Low VRAM headroom")
        print("[RECOMMENDATION] Prefer Batch 8")
    else:
        print("[INFO] Preliminary VRAM headroom available")
        print("[INFO] Longer stability test still needed")

except torch.cuda.OutOfMemoryError:
    print("\n[OOM] Batch 16 exceeded available VRAM")
    print("[RECOMMENDATION] Use Batch 8")
    sys.exit(42)

print("\n[PASS] No checkpoints saved")
print("[PASS] Original P01 untouched")
"""

result = subprocess.run(
    [
        str(PY310),
        "-u",
        "-c",
        script,
        str(ROOT),
        str(DATA),
        str(PRETRAINED),
        str(CONFIG),
    ],
    cwd=ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(result.stdout)

if result.returncode == 42:
    print("\n[RESULT] Batch 16 OOM — Batch 8 preferred")
elif result.returncode == 0:
    print("\n[PASS] STEP 9 COMPLETE")
else:
    raise RuntimeError(
        f"Unexpected failure: exit code {result.returncode}"
    )


## Step 10 — Ten-update real-audio stability smoke test (batch 16)

Run **10 genuine Adam updates on 160 RTCFake Train recordings** with FP32, RawBoost 5 and gradient clipping. This provides stronger memory/finite-value evidence than the single-update tests, but does **not** guarantee a full epoch will fit, and it does not evaluate detection quality.


In [ ]:

from pathlib import Path
import subprocess

ROOT = Path("/kaggle/working/RTC-SDD-Research")
PY310 = Path("/kaggle/working/rtcsdd-baseline-py310/bin/python")
DATA = Path("/kaggle/input/datasets/a7medlabib/rtcfake-packed-private")
PRETRAINED = Path("/kaggle/working/checkpoints/pretrained/xlsr2_300m.pt")
CONFIG = ROOT / "src/rtcsdd/configs/p01_xlsr_aasist_official_like_fp32.json"

for path in [PY310, PRETRAINED, CONFIG, DATA / "train_label.txt"]:
    assert path.is_file(), f"Missing: {path}"

script = r"""
import sys
import json
import time
import math

from pathlib import Path

import torch
from torch.utils.data import DataLoader, Subset

from rtcsdd.data.protocol import read_protocol
from rtcsdd.data.dataset import RTCFakePackedDataset
from rtcsdd.models.xlsr_aasist import load_xlsr_aasist
from rtcsdd.training.trainer import epoch_permutation
from rtcsdd.training.weighted_loss import (
    weighted_ce_sum,
    normalize_gradients,
)

ROOT, DATA, PRETRAINED, CONFIG = map(Path, sys.argv[1:])

cfg = json.loads(CONFIG.read_text())
train = cfg["training"]
data = cfg["data"]

BATCH = 16
ACCUM = 1
UPDATES = 10
DEVICE = "cuda:0"
GIB = 1024 ** 3

print("=" * 70)
print("STEP 10 — BATCH 16 REAL TRAINING STABILITY TEST")
print("=" * 70)

assert torch.cuda.is_available()
assert train["precision"] == "fp32"
assert BATCH * ACCUM == 16
assert data["rawboost"]["enabled"]

torch.cuda.set_device(0)
torch.manual_seed(train["seed"])
torch.cuda.manual_seed_all(train["seed"])
torch.backends.cudnn.benchmark = False

# --------------------------------------------------
# 1. Real training data
# --------------------------------------------------

print("\n[1] PREPARING DATA", flush=True)

examples = read_protocol(DATA / "train_label.txt", "train")

dataset = RTCFakePackedDataset(
    examples,
    DATA,
    length=data["audio_length"],
    crop=data["crop"],
    pad=data["pad"],
    rawboost=data["rawboost"],
    upstream_root=ROOT / "third_party/official_rtc_sdd",
    seed=train["seed"],
)

dataset.set_epoch(0)

indices = epoch_permutation(
    len(dataset),
    train["seed"],
    0,
)[:BATCH * ACCUM * UPDATES]

loader = DataLoader(
    Subset(dataset, indices),
    batch_size=BATCH,
    shuffle=False,
    num_workers=train["workers"],
    pin_memory=True,
    drop_last=False,
)

print("Real recordings :", len(indices))
print("Physical batch  :", BATCH)
print("Accumulation    :", ACCUM)
print("Optimizer steps :", UPDATES)
print("RawBoost        : ON")

# --------------------------------------------------
# 2. Model and optimizer
# --------------------------------------------------

print("\n[2] LOADING MODEL", flush=True)

model = load_xlsr_aasist(
    ROOT / "third_party/official_rtc_sdd",
    PRETRAINED,
    DEVICE,
)

model.train()

parameters = list(model.parameters())

for p in parameters:
    p.requires_grad_(True)

optimizer = torch.optim.Adam(
    parameters,
    lr=train["learning_rate"],
    weight_decay=train["weight_decay"],
)

weights = torch.tensor(
    train["class_weights"],
    device=DEVICE,
    dtype=torch.float32,
)

optimizer.zero_grad(set_to_none=True)

# --------------------------------------------------
# 3. Ten genuine optimizer updates
# --------------------------------------------------

print("\n[3] TRAINING STABILITY", flush=True)

torch.cuda.reset_peak_memory_stats()
started = time.monotonic()

losses = []
grad_norms = []

for step, (audio, labels, ids) in enumerate(loader, 1):

    assert step <= UPDATES
    assert audio.shape == (BATCH, data["audio_length"])
    assert torch.isfinite(audio).all().item()

    audio = audio.to(DEVICE, non_blocking=True)
    labels = labels.to(DEVICE, non_blocking=True)

    logits = model(audio)

    assert torch.isfinite(logits).all().item()

    numerator, denominator = weighted_ce_sum(
        logits,
        labels,
        weights,
    )

    assert torch.isfinite(numerator).item()

    loss_value = (
        numerator.detach() / denominator
    ).item()

    numerator.backward()

    grad_norm = normalize_gradients(
        parameters,
        weight_sum=denominator.item(),
        max_norm=train["gradient_clip_norm"],
    )

    assert math.isfinite(grad_norm), (
        f"Non-finite gradient at step {step}"
    )

    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    torch.cuda.synchronize()

    losses.append(loss_value)
    grad_norms.append(grad_norm)

    peak_reserved = (
        torch.cuda.max_memory_reserved() / GIB
    )

    print(
        f"Step {step:02d}/{UPDATES} | "
        f"Loss={loss_value:.5f} | "
        f"GradNorm={grad_norm:.3f} | "
        f"PeakVRAM={peak_reserved:.2f} GiB",
        flush=True,
    )

    del audio, labels, logits, numerator, denominator

assert step == UPDATES

# --------------------------------------------------
# 4. Results
# --------------------------------------------------

elapsed = time.monotonic() - started

free_bytes, total_bytes = torch.cuda.mem_get_info()

print("\n" + "=" * 70)
print("STABILITY TEST RESULTS")
print("=" * 70)

print("Completed updates :", len(losses))
print("Real audio samples:", len(indices))
print("Precision         : FP32")
print("All losses finite :", all(math.isfinite(x) for x in losses))
print("All norms finite  :", all(math.isfinite(x) for x in grad_norms))
print("Mean Loss         :", f"{sum(losses)/len(losses):.5f}")
print("Total Runtime     :", f"{elapsed:.2f} seconds")
print("Seconds / Update  :", f"{elapsed/UPDATES:.2f}")
print(
    "Peak Allocated    :",
    f"{torch.cuda.max_memory_allocated()/GIB:.2f} GiB"
)
print(
    "Peak Reserved     :",
    f"{torch.cuda.max_memory_reserved()/GIB:.2f} GiB"
)
print("Free VRAM         :", f"{free_bytes/GIB:.2f} GiB")

print("\n[PASS] 10 REAL ADAM UPDATES COMPLETED")
print("[PASS] No checkpoint saved")
print("[PASS] P01 experiment untouched")
print("[INFO] Smoke-test weights discarded on process exit")
"""

subprocess.run(
    [
        str(PY310),
        "-u",
        "-c",
        script,
        str(ROOT),
        str(DATA),
        str(PRETRAINED),
        str(CONFIG),
    ],
    cwd=ROOT,
    check=True,
)

print("\n[PASS] STEP 10 COMPLETE")


## Recorded outcomes — 20 October 2026

The following values were transcribed from the **uploaded notebook's completed outputs**, then the bulky raw stream outputs and ephemeral execution metadata were removed to keep the GitHub notebook reviewable.

### GPU memory diagnostics

| Test | Data | Physical batch × accumulation | Adam step? | Peak reserved GPU VRAM | Free VRAM after test | Outcome |
|---|---|---|---|---:|---:|---|
| Step 6 | Synthetic | 4 × 1 (memory only) | No | 6.24 GiB | 8.18 GiB | PASS |
| Step 7 | Synthetic | 8 × 1 (memory only) | No | 8.25 GiB | 6.17 GiB | PASS |
| Step 7 | Synthetic | 16 × 1 (memory only) | No | 13.24 GiB | 1.18 GiB | PASS |
| Step 8 | Real RTCFake, RawBoost 5 | 8 × 2 | **Yes, 1** | 7.06 GiB | 7.36 GiB | PASS |
| Step 9 | Real RTCFake, RawBoost 5 | 16 × 1 | **Yes, 1** | 10.87 GiB | 3.56 GiB | PASS |
| Step 10 | Real RTCFake, RawBoost 5 | 16 × 1 | **Yes, 10** | **13.33 GiB** | **1.09 GiB** | PASS |

**Important interpretation:** the `1.09 GiB` free GPU memory after ten updates is a **narrow margin**. The ten-step pass does **not** establish that batch 16 is safe for full-epoch training, evaluation or different augmentation workloads. The early one-step result (`10.87 GiB` reserved) understated the later steady peak (`13.33 GiB`). The synthetic tests simulated Adam buffers and did not execute optimizer steps.

### Step 10 — real stability run

| Metric | Recorded value |
|---|---:|
| Completed updates | 10 |
| Audio recordings | 160 |
| Physical batch / accumulation | 16 / 1 |
| Mean per-batch weighted loss | 0.97368 |
| Runtime (10 updates) | 51.23 s |
| Mean wall time per update | 5.12 s |
| Peak allocated / reserved | 12.01 / 13.33 GiB |
| Finite losses / finite gradient norms | Yes / Yes |
| Largest reported pre-clipping gradient norm in this session | 645.212 (step 6) |
| Saved checkpoints | **None** |

**Limitations:** loss values over 160 selected samples are not Dev Macro-F1, Clean/Noisy challenge metrics or evidence of an improved model. They should not be compared as if they were epoch evaluation. This archive does not implement real environmental noise mixing. All original P01 checkpoints remain outside this notebook's generated artifacts.

### Next work (not part of this archived notebook)

- Build shared, configurable runtime/bootstrap commands in the research framework so notebook setup is not copied for each experiment.
- Build a separate generic Git staging/commit/push workflow with explicitly selected repository, branch, file paths and an approval/dry-run step, without exposing credentials.
- Create a fresh experiment notebook for **new-from-pretrained** fine-tuning with RawBoost 5 + audited, permitted non-speech real environmental noise. The later experiment's physical batch and accumulation should be defined in its own config; this archive does not choose them.
